#Avoiding Traps: Leakage - Subjective
**Geetha**

In [1]:
#Python implementation of the flawed baseline approach
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Generate Synthetic Dataset
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)

# 2. Flawed Baseline: Scaling the entire dataset BEFORE splitting
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 4. Train Model
model = LogisticRegression()
model.fit(X_train, y_train)

# 5. Evaluate
train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc = accuracy_score(y_test, model.predict(X_test))

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

Train Accuracy: 0.8700
Test Accuracy: 0.8300


**approach is Data Leakage **(specifically, distribution leakage).

By calling **scaler.fit_transform(X**) on the entire dataset, the StandardScaler calculates the global mean and standard deviation of all 1,000 samples. When the data is later split, the training set effectively "knows" information about the test set's distribution.

#what is wrong in this approach?
**Information Contamination**: The test set is supposed to represent "unseen" future data. In a real-world fintech deployment, you won't have access to the mean or variance of tomorrow's transactions today.

**Over-optimistic Metrics**: Because the training process was influenced by the test set's parameters, the test accuracy may be artificially inflated.

**Reliability**: The model might fail to generalize when deployed because it was tuned—even if subtly—to the specific noise and range of the hold-out data.

#Task 1
Train Accuracy: 0.8588

Test Accuracy: 0.8300
**The Correct Workflow**
To avoid this, you must treat the test set as if it were completely invisible during the training and pre-processing stage:

**Split the data into X_train and X_test**.

**Fit the scaler only on X_train.**

**Transform X_train using those parameters**.

**Transform X_test using the parameters derived from X_train**.

This ensures that the test set remains a "true" hold-out, simulating how the model will handle genuinely new data in the future.

In [2]:
#Task 2
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score

# 1. Split the data FIRST (No scaling yet)
# We use the raw X and y from the make_classification step
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Define the Pipeline
# This ensures scaling happens internally during each CV fold
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])

# 3. Run 5-fold Cross-Validation on the training set
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5)

# 4. Report Results
print(f"Mean CV Accuracy: {cv_scores.mean():.4f}")
print(f"Standard Deviation: {cv_scores.std():.4f}")

# 5. Final check on the unseen test set
pipeline.fit(X_train, y_train)
final_test_acc = pipeline.score(X_test, y_test)
print(f"Final Test Accuracy: {final_test_acc:.4f}")

Mean CV Accuracy: 0.8700
Standard Deviation: 0.0179
Final Test Accuracy: 0.8300


**Task 2: Refactored Pipeline Approach**
When you use a Pipeline inside cross_val_score, the following sequence happens automatically for every fold:

**Isolation**: The data is split into 4 training folds and 1 validation fold.

**Fitting**: The StandardScaler calculates the mean and standard deviation only from the 4 training folds.

**Transformation**: The 4 training folds and the 1 validation fold are scaled using those specific training parameters.

**Training/Evaluation**: The model is trained on the 4 folds and tested on the 1 validation fold.

Results Interpretation

**Results**
**Mean CV Accuracy**: This gives us a much more realistic expectation of how the model will perform on new data. It averages out the performance across different subsets of our data.


---


**Standard Deviation**: This measures the stability of our model. In fintech, a low standard deviation is often just as important as high accuracy; it tells us the model's performance doesn't swing wildly depending on which specific transactions it sees.

In [3]:
#task 3
from sklearn.tree import DecisionTreeClassifier

depths = [1, 5, 20]
results = []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)

    train_acc = dt.score(X_train, y_train)
    test_acc = dt.score(X_test, y_test)

    results.append({"Max Depth": d, "Train Accuracy": train_acc, "Test Accuracy": test_acc})

# Display as a DataFrame or table
import pandas as pd
df_results = pd.DataFrame(results)
print(df_results)

   Max Depth  Train Accuracy  Test Accuracy
0          1         0.88250          0.840
1          5         0.95375          0.855
2         20         1.00000          0.840


#task 3
**Analysis: Balancing Fit and Generalization**
**Depth 1 (Underfitting)**: The model is a "stump." It is too simple to capture the underlying patterns in the dataset, leading to low accuracy on both training and test sets. It has high bias.

**Depth 20 (Overfitting)**: The model has achieved 100% training accuracy, meaning it has essentially memorized every noise and outlier in the training data. However, the test accuracy has dropped compared to Depth 5. This high variance means the model will perform poorly on new, real-world data.

**Depth 5 (Best Balance**): This depth provides the best generalization. It captures enough complexity to improve upon the stump (increasing accuracy by ~5% over Depth 1) without crossing the threshold into memorization

#Conclusion for the Team:
 In our pipeline, **Depth 5 is the superior choice. **It offers the highest test accuracy and the smallest gap between training and testing performance, ensuring that our model remains reliable when evaluating new structured data.